In [0]:

from datetime import datetime, timezone
from delta.tables import DeltaTable

DQ_TABLE = "atliq_dl.silver.dq_log"

def log_dq(table_name, rule, rows_affected):

    run_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    df = spark.createDataFrame(
        [(run_date, table_name, rule, int(rows_affected))],
        "run_date STRING, table_name STRING, rule STRING, rows_affected LONG"
    )

    (
        DeltaTable.forName(spark, DQ_TABLE)
        .alias("t")
        .merge(
            df.alias("s"),
            """t.run_date = s.run_date
               AND t.table_name = s.table_name
               AND t.rule = s.rule"""
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
